# Chapter 7 — TinyCfC: continuous-time gate, training and held-out evaluation

**FlowState Δ · guided educational lab · unexecuted notebook source**

This notebook uses the *existing* `chap-7/cfc_cell.py` and `train_cfc.py` implementation. It is a **CfC-inspired teaching prototype**, not a faithful implementation of the full Hasani et al. benchmark. Execute **Run All** from the repository root or the `chap-7` folder. The cells save *newly measured* figures and JSON rather than embedding invented output.

Reference: R. Hasani et al., [Closed-form continuous-time neural networks](https://doi.org/10.1038/s42256-022-00556-7), *Nature Machine Intelligence*, 2022.

## 1. Analytical bridge and scope

A frozen-coefficient first-order system satisfies `dy/dt = -(y-u)`. For constant input during a time interval, the exact update is

```text
y_next = u + (y_previous - u) * exp(-dt)
```

Our **teaching CfC-style** update is `alpha = sigmoid(-f(x,u)*dt)` and `x_next = alpha*g(x,u) + (1-alpha)*h(x,u)`. The heads and backbone are learned; the numerical integral is **not** being optimized. A short-sequence training fit is not evidence of generalization or solver equivalence.

In [ ]:
from pathlib import Path
import sys, json, torch
import matplotlib.pyplot as plt

cwd = Path.cwd()
ch7 = cwd if (cwd / "train_cfc.py").exists() else cwd / "chap-7"
assert (ch7 / "train_cfc.py").exists(), "Start in repository root or chap-7/"
sys.path.insert(0, str(ch7))
from train_cfc import SequenceCfC
from cfc_cell import TinyCfC

torch.set_num_threads(1)
print("PyTorch:", torch.__version__)
print("Chapter 7 path:", ch7.resolve())
print("Existing cfc_cell.py may print its import-time smoke-test diagnostics.")

## 2. Check analytic target and time-gate behavior

In [ ]:
dt_short, dt_long = torch.tensor(0.1), torch.tensor(2.0)
short = 1.0 - torch.exp(-dt_short)
long = 1.0 - torch.exp(-dt_long)
assert torch.isclose(short, torch.tensor(0.09516258), atol=1e-6)
assert torch.isclose(long, torch.tensor(0.86466472), atol=1e-6)
print("Analytical one-step responses:", short.item(), long.item())

torch.manual_seed(42)
cell = TinyCfC()
state = torch.zeros(1, 4)
input_value = torch.ones(1, 1)
with torch.no_grad():
    y_short, alpha_short = cell(state, input_value, torch.tensor([[0.1]]))
    y_long, alpha_long = cell(state, input_value, torch.tensor([[2.0]]))
print("Short/long gates:", alpha_short, alpha_long)
print("Short/long states:", y_short, y_long)
assert y_short.shape == y_long.shape == (1, 4)

## 3. Rerun a small training and one held-out sequence

The targets in the original `train_cfc.py` are rounded to four decimal places. We deliberately keep that convention here. **New Run All outputs may differ** from numbers recorded in earlier chat reports because initialization/training context can differ. We record the actual measurements rather than forcing agreement.

In [ ]:
inputs = torch.tensor([0., 1., 1., 0., 1., 0.])
dts = torch.tensor([0.1, 0.5, 1.0, 0.25, 0.8, 0.2])
targets = torch.tensor([0., 0.3935, 0.7769, 0.6050, 0.8225, 0.6734])

torch.manual_seed(42)
model = SequenceCfC()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_curve = []
for step in range(101):
    optimizer.zero_grad(set_to_none=True)
    predicted = model(inputs, dts)
    loss = (predicted - targets).square().mean()
    loss.backward()
    optimizer.step()
    loss_curve.append(loss.item())

test_inputs = torch.tensor([1., 0., 1., 1., 0., 0., 1., 0.])
test_dts = torch.tensor([0.3, 0.75, 0.45, 1.3, 0.1, 0.9, 0.6, 0.4])
system_state = torch.tensor(0.0)
test_targets = []
for u, dt in zip(test_inputs, test_dts):
    system_state = u + (system_state - u) * torch.exp(-dt)
    test_targets.append(system_state)
test_targets = torch.stack(test_targets)

model.eval()
with torch.no_grad():
    test_pred = model(test_inputs, test_dts)
    train_mse = (model(inputs, dts) - targets).square().mean().item()
    test_mse = (test_pred - test_targets).square().mean().item()
    baseline_value = targets.mean().item()
    baseline_pred = torch.full_like(test_targets, baseline_value)
    baseline_mse = (baseline_pred - test_targets).square().mean().item()

assert torch.isfinite(test_pred).all()
assert len(test_pred) == 8
print(dict(train_mse=train_mse, heldout_mse=test_mse,
           constant_heldout_mse=baseline_mse,
           parameters=sum(p.numel() for p in model.parameters())))

## 4. Export **fresh** figures and metadata

The `figures/*.svg` files already in GitHub are historical/provenance-labelled documentation figures. This cell creates separate files with `rerun_` names based on **this notebook's actual execution**.

In [ ]:
fig_dir = ch7 / "figures"
result_dir = ch7 / "results"
fig_dir.mkdir(exist_ok=True)
result_dir.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(range(len(loss_curve)), loss_curve)
ax.set(xlabel="Optimizer update", ylabel="Training MSE", title="TinyCfC fresh training run")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(fig_dir / "rerun_training_loss.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(test_targets.numpy(), marker="o", label="Analytical target")
ax.plot(test_pred.detach().numpy(), marker="s", label="TinyCfC prediction")
ax.plot(baseline_pred.numpy(), linestyle="--", label="Training-mean baseline")
ax.set(xlabel="Held-out time step", ylabel="Output", title="Fresh held-out evaluation")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(fig_dir / "rerun_heldout.png", dpi=150)
plt.show()

metrics = dict(seed=42, train_steps=101, train_points=6, heldout_points=8,
               train_mse=train_mse, heldout_mse=test_mse,
               constant_heldout_mse=baseline_mse, params=sum(p.numel() for p in model.parameters()),
               provenance="fresh notebook execution", publication_reproduction=False)
(result_dir / "ch7_notebook_rerun.json").write_text(json.dumps(metrics, indent=2) + "\n")
print("Saved fresh figures and metrics to", fig_dir, result_dir)

## 5. Interpretation and student checks

1. Why can a model have very low training MSE while performing worse than a constant baseline on held-out points?
2. For a *fixed positive* f, what happens to `alpha = sigmoid(-f*dt)` as dt increases? What if the learned f is negative?
3. What is missing before interpreting a CfC-versus-LTC speed/accuracy comparison as fair?

**Limitations:** one small synthetic task; one initialization; no multi-seed uncertainty; model is not the full published CfC; the simple constant baseline exploits the training mean; no production efficiency claims. A separate Chapter 1-adapted comparison is available in `compare_ch1_ch7.py`, but it is **not** parameter-matched and is **not** the original November 15 local script. Chapter 8 will define stronger comparisons.